# Official end-to-end smoke notebook

This is the durable, official end-to-end smoke notebook for `geecomposer`.

It exercises the public API surface against a real Earth Engine session and
the case-study AOI at `01_data/case_studies/rbmn.geojson`.

What this notebook covers:

1. Earth Engine initialization
2. AOI normalization from a local GeoJSON file
3. Sentinel-2 single-band median composite (with cloud masking)
4. Sentinel-2 NDVI max composite (with cloud masking + transform)
5. Sentinel-1 dB VV median composite (radar path, no preprocessing)
6. Yearly grouping over Sentinel-1 dB
7. Drive export task creation, start, and polling
8. A clean run summary

What this notebook does **not** cover:

- Sentinel-1 float + Gamma MAP speckle filtering. That workflow has its
  own dedicated notebook: `006_sentinel1_float_gamma_map_smoke.ipynb`.
- Numerical correctness of EE outputs. This notebook is a smoke test
  (does the workflow run end-to-end?), not a scientific validation.

## Live-validation assumptions

- **Authentication:** assumes EE credentials are already available
  locally (`authenticate=False`). On a new machine, change the cell
  below to `initialize(project=..., authenticate=True)` and complete
  the interactive flow once.
- **Project:** hardcodes `project="manglariars"` for the active
  development project. Change to your own EE-enabled GCP project if
  reusing this notebook outside Manglaria.
- **Drive folder:** exports go to `geecomposer-dev` by default. Rename
  freely. The folder is created on first export if missing.
- **Task start:** the export cell is gated behind `START_EXPORT = False`
  so re-running this notebook does not silently launch a new EE batch
  task. Flip to `True` only when you intend to start an export.


## 1. Path setup and imports

We resolve the repo root robustly so the notebook can be run from any
working directory inside the repo (VS Code, JupyterLab, classic
Jupyter).


In [ ]:
from pathlib import Path
import sys

import ee


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "geecomposer_v0.1_spec.md").exists() and (
            candidate / "08_pkg" / "src"
        ).exists():
            return candidate
    raise RuntimeError("Could not find repo root from current working directory.")


repo_root = find_repo_root(Path.cwd().resolve())
pkg_src = repo_root / "08_pkg" / "src"
aoi_path = repo_root / "01_data" / "case_studies" / "rbmn.geojson"

if str(pkg_src) not in sys.path:
    sys.path.insert(0, str(pkg_src))

from geecomposer import compose, compose_yearly, export_to_drive, initialize
from geecomposer.aoi import to_ee_geometry
from geecomposer.transforms.indices import ndvi

print("Repo root:    ", repo_root)
print("Package src:  ", pkg_src)
print("AOI path:     ", aoi_path)


## 2. Initialize Earth Engine

Set `authenticate=True` once if local credentials are not yet
available. Subsequent runs should use `authenticate=False`.


In [ ]:
initialize(project="manglariars", authenticate=False)

# Liveness check: a trivial server-side computation.
assert ee.Number(1).getInfo() == 1
print("Earth Engine initialized.")


## 3. Load the case-study AOI

The AOI is a polygon over the Marismas Nacionales mangrove system.
`to_ee_geometry()` accepts the path directly; we materialize the
bounds purely as a quick eyeball check.


In [ ]:
aoi = to_ee_geometry(aoi_path)
aoi.bounds().getInfo()


## 4. Sentinel-2 red-band median composite

A minimal optical workflow: cloud-masked B4 median over the calendar
year. This validates the dataset preset, masking preset, band
selection, reducer, and metadata path together.


In [ ]:
s2_red = compose(
    dataset="sentinel2",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    mask="s2_cloud_score_plus",
    select="B4",
    reducer="median",
)

s2_red_bands = s2_red.bandNames().getInfo()
s2_red_props = s2_red.getInfo()["properties"]

assert s2_red_bands == ["B4"]
assert s2_red_props["geecomposer:dataset"] == "sentinel2"
assert s2_red_props["geecomposer:reducer"] == "median"

{"bands": s2_red_bands, "properties": s2_red_props}


## 5. Sentinel-2 NDVI max composite

Same dataset and AOI, but with the built-in `ndvi()` transform applied
per image before the temporal reduction. The transform name is
recorded in metadata so downstream consumers can identify the recipe.


In [ ]:
s2_ndvi = compose(
    dataset="sentinel2",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    mask="s2_cloud_score_plus",
    transform=ndvi(),
    reducer="max",
)

s2_ndvi_bands = s2_ndvi.bandNames().getInfo()
s2_ndvi_props = s2_ndvi.getInfo()["properties"]

assert s2_ndvi_bands == ["ndvi"]
assert s2_ndvi_props["geecomposer:transform"] == "ndvi"

{"bands": s2_ndvi_bands, "properties": s2_ndvi_props}


## 6. Sentinel-1 dB VV median composite

Canonical radar path. `dataset="sentinel1"` returns dB-scaled
backscatter; for ratio features in linear power use
`dataset="sentinel1_float"` (covered in
`006_sentinel1_float_gamma_map_smoke.ipynb`).


In [ ]:
s1_vv = compose(
    dataset="sentinel1",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    select="VV",
    reducer="median",
    filters={
        "instrumentMode": "IW",
        "polarizations": ["VV"],
        "orbitPass": "ASCENDING",
    },
)

s1_vv_bands = s1_vv.bandNames().getInfo()
s1_vv_props = s1_vv.getInfo()["properties"]

assert s1_vv_bands == ["VV"]
assert s1_vv_props["geecomposer:dataset"] == "sentinel1"
assert s1_vv_props["geecomposer:reducer"] == "median"

{"bands": s1_vv_bands, "properties": s1_vv_props}


## 7. Yearly grouping

`compose_yearly()` returns one `ee.Image` per year. Date ranges are
derived as `{year}-01-01` to `{year+1}-01-01` (inclusive start /
exclusive end, matching `filterDate`).


In [ ]:
yearly_s1_vv = compose_yearly(
    years=[2023, 2024, 2025],
    dataset="sentinel1",
    aoi=str(aoi_path),
    select="VV",
    reducer="median",
    filters={"instrumentMode": "IW", "polarizations": ["VV"]},
)

assert set(yearly_s1_vv.keys()) == {2023, 2024, 2025}

year_2025_props = yearly_s1_vv[2025].getInfo()["properties"]
{"years": sorted(yearly_s1_vv.keys()), "properties_2025": year_2025_props}


## 8. Drive export — create a task

`export_to_drive()` creates the task but does **not** start it. This
lets you inspect the task object and pick a unique description before
launching anything.

Re-using the same description across runs can produce
"A different operation was already started with the same request id"
errors. Bump the description suffix (`_v2`, `_v3`, ...) when re-running.


In [ ]:
drive_folder = "geecomposer-dev"
export_description = "rbmn_s1_vv_median_2025_smoke"

s1_2025_task = export_to_drive(
    image=yearly_s1_vv[2025],
    description=export_description,
    folder=drive_folder,
    region=str(aoi_path),
    scale=10,
)

s1_2025_task.status()


## 9. Drive export — start (gated)

The export is gated behind `START_EXPORT = False` so re-running the
notebook never silently launches a new EE batch task. Flip the flag
to `True` only when you actually want to export.

Cost note: a year of Sentinel-1 IW VV over the case-study AOI at 10 m
typically takes a few minutes of EE batch compute and produces a
multi-MB GeoTIFF in your Drive folder.


In [ ]:
START_EXPORT = False

if START_EXPORT:
    s1_2025_task.start()
    print("Export task started.")
else:
    print("Skipping start. Set START_EXPORT = True to launch.")

s1_2025_task.status()


## 10. Drive export — poll until completion

Run this cell only after starting the task. It polls every
`SLEEP_SECONDS` until a terminal state or `MAX_POLLS` is reached.
Terminal states: `COMPLETED`, `FAILED`, `CANCELLED`.


In [ ]:
import time
from pprint import pprint

MAX_POLLS = 100
SLEEP_SECONDS = 20

if not START_EXPORT:
    print("Skipping polling because START_EXPORT is False.")
else:
    for i in range(MAX_POLLS):
        status = s1_2025_task.status()
        state = status.get("state", "UNKNOWN")
        print(f"Poll {i + 1}/{MAX_POLLS}: {state}")
        if state in {"COMPLETED", "FAILED", "CANCELLED"}:
            pprint(status)
            break
        time.sleep(SLEEP_SECONDS)
    else:
        print("Polling limit reached; task may still be running.")
        pprint(s1_2025_task.status())


## 11. Run summary

Lightweight summary of what this notebook produced. Uses only
variables that exist in the current kernel session, so re-running the
notebook from scratch always yields a coherent summary.


In [ ]:
summary = {
    "aoi_path": str(aoi_path),
    "sentinel2_red_bands": s2_red_bands,
    "sentinel2_ndvi_bands": s2_ndvi_bands,
    "sentinel1_vv_bands": s1_vv_bands,
    "yearly_s1_vv_keys": sorted(yearly_s1_vv.keys()),
    "export_description": export_description,
    "export_started": START_EXPORT,
    "export_state": s1_2025_task.status().get("state"),
}
summary
